# Baseline hafalan kata
**text › ner** · notebook `majority_baseline`

**Kapan dipakai:** Batas bawah: jika model Anda hampir tidak mengalahkannya, ada yang salah.

**Jebakan umum:** Kata baru selalu 'O'.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/text/ner/nerp_train.csv"
TEST_PATH  = ROOT / "data_latihan/text/ner/nerp_test.csv"      # berlabel; dipakai sebagai evaluasi akhir (JANGAN dipakai untuk tuning)
MODEL_KEY  = "majority_baseline"
OUT = out_dir("text", "ner", MODEL_KEY)
set_seed(42)

## 2. Baca data
Format CSV: `sentence_id, token, tag` dengan tag BIO (B-X awal entitas, I-X lanjutan, O bukan entitas).

In [ ]:
from superdb.text_extra import load_ner_csv, sent_features, entity_f1, entity_spans
tr_tok, tr_tag = load_ner_csv(TRAIN_PATH); te_tok, te_tag = load_ner_csv(TEST_PATH)
print(len(tr_tok), "kalimat latih,", len(te_tok), "kalimat uji")
display(pd.Series([t for s in tr_tag for t in s]).value_counts().to_frame("jumlah token").T)
print(list(zip(tr_tok[0], tr_tag[0]))[:12])

## 3. Tagger
Semua tagger berbagi API: `.fit(fitur_kalimat, tag_kalimat)` dan `.predict(fitur_kalimat)`.

In [ ]:
from superdb.text_extra import MajorityTagger

MODE = "basic"
tagger = MajorityTagger()
Xtr_f = [sent_features(s, MODE) for s in tr_tok]; Xte_f = [sent_features(s, MODE) for s in te_tok]
tagger.fit(Xtr_f, tr_tag)
pred = tagger.predict(Xte_f)

## 4. Evaluasi level entitas
Entitas dihitung benar hanya jika **batas dan tipenya persis sama**. Ini lebih ketat dan lebih relevan daripada akurasi per token.

In [ ]:
micro, per_type = entity_f1(te_tag, pred)
tok_acc = np.mean([a == b for ys, ps in zip(te_tag, pred) for a, b in zip(ys, ps)])
print({k: round(v, 4) for k, v in micro.items()}, "| akurasi token:", round(float(tok_acc), 4))
display(per_type.round(3))

## 5. Contoh kesalahan

In [ ]:
shown = 0
for toks, yt, yp in zip(te_tok, te_tag, pred):
    a, b = entity_spans(yt), entity_spans(yp)
    if a != b and shown < 5:
        fmt = lambda S: [(" ".join(toks[s:e]), t) for s, e, t in sorted(S)]
        print("KALIMAT :", " ".join(toks)[:160]); print("  benar   :", fmt(a)); print("  prediksi:", fmt(b)); shown += 1

## 6. Simpan & uji muat ulang

In [ ]:
save_model({"tagger": tagger, "mode": MODE}, OUT / "model.joblib")
save_json({"entity_micro": micro, "token_accuracy": float(tok_acc), "per_type": per_type.round(4).to_dict("index")}, OUT / "metrics.json")
b = load_model(OUT / "model.joblib"); assert [list(x) for x in b["tagger"].predict(Xte_f[:20])] == [list(x) for x in tagger.predict(Xte_f[:20])]; print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan CRF dengan Logistic Regression per token: selisihnya = manfaat memodelkan transisi tag.
- Entitas yang sering salah batas: tambah fitur konteks atau gazetteer (daftar nama).
- Format output lomba (JSON/CoNLL): ubah `pred` per kalimat sesuai format.